In [0]:
silver_base = "abfss://shopsphere@stshopsphere2026.dfs.core.windows.net/silver"

customers_silver_df = spark.read.format("delta").load(f"{silver_base}/customers")
orders_silver_df = spark.read.format("delta").load(f"{silver_base}/orders")
products_silver_df = spark.read.format("delta").load(f"{silver_base}/products")


In [0]:
from pyspark.sql.functions import col, count

def run_dq_check(check_name, condition):
    failed_count = condition.count()
    
    status = "PASS" if failed_count == 0 else "FAIL"
    
    print(f"{check_name}: {status} | Failed rows: {failed_count}")
    
    return failed_count

In [0]:
customer_failures = 0

customer_failures += run_dq_check(
    "Customer ID NULL",
    customers_silver_df.filter(col("customer_id").isNull())
)

customer_failures += run_dq_check(
    "Duplicate Customer ID",
    customers_silver_df
        .groupBy("customer_id")
        .count()
        .filter(col("count") > 1)
)

Customer ID NULL: PASS | Failed rows: 0
Duplicate Customer ID: PASS | Failed rows: 0


In [0]:
product_failures = 0

product_failures += run_dq_check(
    "Product ID NULL",
    products_silver_df.filter(col("product_id").isNull())
)

product_failures += run_dq_check(
    "Duplicate Product ID",
    products_silver_df
        .groupBy("product_id")
        .count()
        .filter(col("count") > 1)
)

product_failures += run_dq_check(
    "Invalid Price",
    products_silver_df.filter(
        col("price").isNull() | (col("price") <= 0)
    )
)

product_failures += run_dq_check(
    "Invalid Cost",
    products_silver_df.filter(
        col("cost").isNull() | (col("cost") < 0)
    )
)

product_failures += run_dq_check(
    "Invalid Stock",
    products_silver_df.filter(
        col("stock_quantity").isNull() |
        (col("stock_quantity") < 0)
    )
)

Product ID NULL: PASS | Failed rows: 0
Duplicate Product ID: PASS | Failed rows: 0
Invalid Price: PASS | Failed rows: 0
Invalid Cost: PASS | Failed rows: 0
Invalid Stock: PASS | Failed rows: 0


In [0]:
order_failures = 0

order_failures += run_dq_check(
    "Order ID NULL",
    orders_silver_df.filter(col("order_id").isNull())
)

order_failures += run_dq_check(
    "Duplicate Order ID",
    orders_silver_df
        .groupBy("order_id")
        .count()
        .filter(col("count") > 1)
)

order_failures += run_dq_check(
    "Invalid Quantity",
    orders_silver_df.filter(
        col("quantity").isNull() | (col("quantity") <= 0)
    )
)

order_failures += run_dq_check(
    "Invalid Discount",
    orders_silver_df.filter(
        col("discount").isNull() |
        (col("discount") < 0) |
        (col("discount") > 1)
    )
)

order_failures += run_dq_check(
    "Invalid Total Amount",
    orders_silver_df.filter(
        col("total_amount").isNull() |
        (col("total_amount") < 0)
    )
)

order_failures += run_dq_check(
    "Missing Order Date",
    orders_silver_df.filter(col("order_date").isNull())
)

Order ID NULL: PASS | Failed rows: 0
Duplicate Order ID: PASS | Failed rows: 0
Invalid Quantity: PASS | Failed rows: 0
Invalid Discount: PASS | Failed rows: 0
Invalid Total Amount: PASS | Failed rows: 0
Missing Order Date: PASS | Failed rows: 0


In [0]:
invalid_customer_orders = (
    orders_silver_df
    .join(
        customers_silver_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Invalid Customer References:",
    invalid_customer_orders.count()
)

Invalid Customer References: 0


In [0]:
invalid_product_orders = (
    orders_silver_df
    .join(
        products_silver_df.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print(
    "Invalid Product References:",
    invalid_product_orders.count()
)

Invalid Product References: 0


In [0]:
total_failures = (
    customer_failures +
    product_failures +
    order_failures +
    invalid_customer_orders.count() +
    invalid_product_orders.count()
)

print("=" * 50)

if total_failures == 0:
    print("DATA QUALITY STATUS: PASS")
    print("All critical checks passed.")
else:
    print("DATA QUALITY STATUS: FAIL")
    print(f"Total failed checks/rows: {total_failures}")
    raise Exception("Data Quality checks failed")

DATA QUALITY STATUS: PASS
All critical checks passed.


In [0]:
print("=" * 50)
print("SHOPSPHERE DATA QUALITY")
print("=" * 50)

print(f"Customers checked: {customers_silver_df.count()}")
print(f"Products checked:  {products_silver_df.count()}")
print(f"Orders checked:    {orders_silver_df.count()}")

if total_failures == 0:
    print("DQ STATUS: PASS")
else:
    print("DQ STATUS: FAIL")

SHOPSPHERE DATA QUALITY
Customers checked: 1001
Products checked:  104
Orders checked:    9339
DQ STATUS: PASS
